# Qwen3 0.6B + Mimi Codes Stage 1 Training

This notebook trains a **Stage 1 audio-token model** using:

```text
base model: Qwen/Qwen3-0.6B
codec: Kyutai Mimi
input dataset columns: id, text, speaker_id, codes, n_frames, k_codebooks
codes shape: [8, n_frames]
```

This is not a text-only chat SFT notebook anymore. It uses a **real-time-oriented frame architecture**:

```text
Qwen3 0.6B temporal backbone
+ previous Mimi frame embeddings
+ 8 parallel codebook heads
```

That means the large Qwen model runs once per Mimi frame, not once per codebook token. Mimi runs at 12.5 frames/sec, so this is the architecture direction needed for real-time speech.

Stage 1 goal:

```text
text transcript + previous audio frames -> next Mimi frame cb0-cb7
```

Stage 1 does **not** teach full-duplex dialogue yet. It teaches Qwen to model Mimi speech codes. Stage 2/3 will add clean conversation and overlap behavior.

## Is Real-Time Possible With This Dataset?

Yes, as a **Stage 1 foundation**.

The dataset gives all 8 Mimi codebooks at 12.5 fps:

```text
codes: [8, n_frames]
```

A bad flattened-token approach would make Qwen generate:

```text
12.5 frames/sec * 8 codebooks = 100 autoregressive tokens/sec
```

This notebook instead trains the better real-time pattern:

```text
one Qwen temporal step per frame
8 codebook heads predict cb0-cb7 together
```

So the big model operates near:

```text
12.5 steps/sec
```

Real-time full duplex still requires later training stages:

```text
Stage 1: LibriSpeech Mimi speech-code pretraining
Stage 2: clean two-speaker ZipVoice dialogue
Stage 3: overlap/backchannel/interruption training
```

In [ ]:
# Install dependencies. Run this once per fresh machine/runtime.
# On a VM with packages already installed, you can skip this cell.

# Keep the machine's CUDA-matched Torch install. Install PyTorch separately only if missing.
%pip install -U "transformers>=4.56.0" "datasets>=2.20.0" "accelerate>=0.33.0" "huggingface_hub>=0.34.0" "safetensors>=0.4.3"

In [ ]:
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

set_seed(3407)

MODEL_NAME = "Qwen/Qwen3-0.6B"
DATASET_NAME = "shangeth/librispeech-mimi-codes"
SPLIT = "train_clean_100"

OUTPUT_DIR = Path("qwen3_0_6b_mimi_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_CODEBOOKS = 8
CODEBOOK_SIZE = 2048
START_CODE = CODEBOOK_SIZE  # extra input-only start code for previous-frame conditioning

# Start small. Increase after the loss and generated codes look sane.
TRAIN_LIMIT = 2_000        # None for full split
EVAL_LIMIT = 128
MAX_TEXT_TOKENS = 192
MAX_FRAMES = 192           # 192 frames ~= 15.36 sec at 12.5 fps

# Qwen 0.6B is small enough to train last layers on a good GPU.
TRAIN_LAST_N_LAYERS = 4    # set 0 to freeze Qwen and train only audio embeddings/heads
LEARNING_RATE = 2e-4
MAX_STEPS = 1_000
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8

print("model:", MODEL_NAME)
print("dataset:", DATASET_NAME, SPLIT)
print("output:", OUTPUT_DIR)

## Load Dataset

This notebook defaults to `shangeth/librispeech-mimi-codes`, which matches the sample row `1272-128104-0000` shown above. You can swap to `shangeth/libritts-r-mimi-codes` later if you want punctuation-preserved TTS-style text.

Expected row schema:

```text
id: string
text: string
speaker_id: int32
codes: list [8][n_frames]
n_frames: int32
k_codebooks: int32
```

The notebook truncates long examples to `MAX_FRAMES` so the first run does not blow up memory.

In [ ]:
raw = load_dataset(DATASET_NAME, split=SPLIT)
print(raw)
print(raw.column_names)
print(raw[0].keys())
print("first text:", raw[0]["text"])
print("first n_frames:", raw[0]["n_frames"], "k:", raw[0]["k_codebooks"])

if TRAIN_LIMIT is not None:
    total = min(len(raw), TRAIN_LIMIT + EVAL_LIMIT)
    raw = raw.select(range(total))

split = raw.train_test_split(test_size=min(EVAL_LIMIT, max(1, len(raw) // 20)), seed=3407)
train_raw = split["train"]
eval_raw = split["test"]

print("train rows:", len(train_raw))
print("eval rows:", len(eval_raw))

## Tokenizer And Qwen Backbone

We do **not** add 16K Mimi tokens to the Qwen tokenizer in this notebook.

Instead:

```text
text uses normal Qwen tokenizer
Mimi codes use separate codebook embeddings and output heads
```

That is closer to the real-time architecture and avoids inflating the text vocabulary.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
bf16_ok = bool(use_cuda and torch.cuda.is_bf16_supported())
torch_dtype = torch.bfloat16 if bf16_ok else (torch.float16 if use_cuda else torch.float32)

base_lm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
)
base_lm.config.use_cache = False

print("dtype:", torch_dtype)
print("hidden size:", base_lm.config.hidden_size)
print("layers:", len(base_lm.model.layers))
print("vocab:", len(tokenizer))

## Dataset Wrapper And Collator

The collator returns:

```text
input_ids: normal Qwen text prompt tokens
attention_mask: text mask
codes: [batch, 8, frames], padded with -100
code_mask: [batch, frames]
```

The model shifts `codes` internally:

```text
previous frame -> predict current frame
```

In [ ]:
class MimiCodeRows(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        return {
            "id": row["id"],
            "text": row["text"],
            "codes": row["codes"],
            "n_frames": row["n_frames"],
            "k_codebooks": row["k_codebooks"],
        }


@dataclass
class MimiFrameCollator:
    tokenizer: AutoTokenizer
    max_text_tokens: int = MAX_TEXT_TOKENS
    max_frames: int = MAX_FRAMES
    k_codebooks: int = K_CODEBOOKS

    def __call__(self, rows):
        prompts = [self._format_prompt(row["text"]) for row in rows]
        text_batch = self.tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=self.max_text_tokens,
            return_tensors="pt",
            add_special_tokens=True,
        )

        frame_count = min(
            self.max_frames,
            max(min(int(row["n_frames"]), len(row["codes"][0])) for row in rows),
        )
        codes = torch.full(
            (len(rows), self.k_codebooks, frame_count),
            fill_value=-100,
            dtype=torch.long,
        )
        code_mask = torch.zeros((len(rows), frame_count), dtype=torch.long)

        for batch_idx, row in enumerate(rows):
            if int(row["k_codebooks"]) < self.k_codebooks:
                raise ValueError(f"row {row['id']} has only {row['k_codebooks']} codebooks")
            row_codes = torch.tensor(row["codes"][: self.k_codebooks], dtype=torch.long)
            if row_codes.ndim != 2 or row_codes.shape[0] != self.k_codebooks:
                raise ValueError(f"row {row['id']} codes must be [8, n_frames], got {tuple(row_codes.shape)}")
            n = min(frame_count, row_codes.shape[1], self.max_frames)
            codes[batch_idx, :, :n] = row_codes[:, :n]
            code_mask[batch_idx, :n] = 1

        return {
            "input_ids": text_batch["input_ids"],
            "attention_mask": text_batch["attention_mask"],
            "codes": codes,
            "code_mask": code_mask,
        }

    @staticmethod
    def _format_prompt(text: str) -> str:
        # Keep this simple for Stage 1. Later stages can add speaker/stream tags.
        return f"<speech_text> {text.strip()}\n<audio_codes>"


train_dataset = MimiCodeRows(train_raw)
eval_dataset = MimiCodeRows(eval_raw)
collator = MimiFrameCollator(tokenizer)

batch = collator([train_dataset[0], train_dataset[1]])
for key, value in batch.items():
    print(key, tuple(value.shape), value.dtype)

## Qwen + Mimi Frame Model

This model uses Qwen as a temporal backbone.

For each audio frame:

```text
input to frame position: previous frame cb0-cb7 embeddings
output from frame position: logits for current frame cb0-cb7
```

The large transformer sees one position per Mimi frame. The 8 codebook heads predict the full frame.

In [ ]:
class QwenMimiFrameModel(nn.Module):
    def __init__(
        self,
        qwen_lm: AutoModelForCausalLM,
        k_codebooks: int = K_CODEBOOKS,
        codebook_size: int = CODEBOOK_SIZE,
        start_code: int = START_CODE,
        cb0_weight: float = 8.0,
    ):
        super().__init__()
        self.transformer = qwen_lm.model
        self.text_embeddings = qwen_lm.get_input_embeddings()
        self.config = qwen_lm.config
        self.k_codebooks = k_codebooks
        self.codebook_size = codebook_size
        self.start_code = start_code
        hidden_size = qwen_lm.config.hidden_size

        self.frame_marker = nn.Parameter(torch.zeros(hidden_size))
        self.codebook_embeddings = nn.ModuleList(
            [nn.Embedding(codebook_size + 1, hidden_size) for _ in range(k_codebooks)]
        )
        self.codebook_heads = nn.ModuleList(
            [nn.Linear(hidden_size, codebook_size) for _ in range(k_codebooks)]
        )
        weights = torch.ones(k_codebooks)
        weights[0] = cb0_weight
        self.register_buffer("codebook_loss_weights", weights, persistent=False)

        self._init_audio_modules()

    def _init_audio_modules(self):
        nn.init.normal_(self.frame_marker, mean=0.0, std=0.02)
        for emb in self.codebook_embeddings:
            nn.init.normal_(emb.weight, mean=0.0, std=0.02)
        for head in self.codebook_heads:
            nn.init.normal_(head.weight, mean=0.0, std=0.02)
            nn.init.zeros_(head.bias)

    def freeze_backbone_except_last_layers(self, last_n_layers: int):
        for param in self.transformer.parameters():
            param.requires_grad = False
        if last_n_layers > 0:
            for layer in self.transformer.layers[-last_n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
            if hasattr(self.transformer, "norm"):
                for param in self.transformer.norm.parameters():
                    param.requires_grad = True

    def build_prev_codes(self, codes: torch.Tensor) -> torch.Tensor:
        # codes: [B, K, T], padded labels are -100.
        safe_codes = codes.clamp(min=0)
        prev_codes = torch.full_like(safe_codes, fill_value=self.start_code)
        prev_codes[:, :, 1:] = safe_codes[:, :, :-1]
        prev_codes = torch.where(codes.eq(-100), torch.full_like(prev_codes, self.start_code), prev_codes)
        return prev_codes

    def frame_inputs_from_prev_codes(self, prev_codes: torch.Tensor) -> torch.Tensor:
        # prev_codes: [B, K, T], values 0..2048 where 2048 is start/input-only.
        frame_emb = self.frame_marker.view(1, 1, -1).to(prev_codes.device)
        pieces = []
        for cb in range(self.k_codebooks):
            pieces.append(self.codebook_embeddings[cb](prev_codes[:, cb, :]))
        stacked = torch.stack(pieces, dim=0).sum(dim=0) / math.sqrt(self.k_codebooks)
        return stacked + frame_emb

    def forward_logits(self, input_ids, attention_mask, prev_codes, code_mask):
        text_emb = self.text_embeddings(input_ids)
        frame_emb = self.frame_inputs_from_prev_codes(prev_codes).to(dtype=text_emb.dtype)
        inputs_embeds = torch.cat([text_emb, frame_emb], dim=1)
        full_attention_mask = torch.cat([attention_mask, code_mask], dim=1)
        outputs = self.transformer(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attention_mask,
            use_cache=False,
        )
        audio_hidden = outputs.last_hidden_state[:, input_ids.shape[1] :, :]
        return [head(audio_hidden.to(dtype=head.weight.dtype)) for head in self.codebook_heads]

    def forward(self, input_ids, attention_mask, codes, code_mask):
        prev_codes = self.build_prev_codes(codes)
        logits_by_cb = self.forward_logits(input_ids, attention_mask, prev_codes, code_mask)
        total_loss = None
        total_weight = 0.0
        for cb, logits in enumerate(logits_by_cb):
            labels = codes[:, cb, :].contiguous()
            loss = F.cross_entropy(
                logits.reshape(-1, self.codebook_size).float(),
                labels.reshape(-1),
                ignore_index=-100,
            )
            weight = self.codebook_loss_weights[cb].float()
            total_loss = loss * weight if total_loss is None else total_loss + loss * weight
            total_weight += float(weight.item())
        total_loss = total_loss / total_weight
        return {"loss": total_loss, "logits": logits_by_cb[0]}


model = QwenMimiFrameModel(base_lm)
model.freeze_backbone_except_last_layers(TRAIN_LAST_N_LAYERS)
base_lm = None  # the wrapped Qwen module is now owned by model

def count_trainable_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_trainable_params(model)
print(f"total params: {total/1e6:.1f}M")
print(f"trainable params: {trainable/1e6:.1f}M ({trainable/total:.2%})")

## Train

Default settings are intentionally small. First run should prove:

```text
loss decreases
cb0 is not collapsed
checkpoints save correctly
```

Then increase:

```text
TRAIN_LIMIT
MAX_STEPS
MAX_FRAMES
TRAIN_LAST_N_LAYERS
```

In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16:", torch.cuda.is_bf16_supported())
else:
    print("CUDA not available. This notebook is intended for GPU training.")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=250,
    save_total_limit=2,
    bf16=bf16_ok,
    fp16=bool(torch.cuda.is_available() and not bf16_ok),
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    gradient_checkpointing=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()

## Save Checkpoint

This is a custom architecture, so save the PyTorch state plus metadata.

For production training, move this model class into the repo as a normal module and save/load with a dedicated config.

In [ ]:
final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "split": SPLIT,
        "k_codebooks": K_CODEBOOKS,
        "codebook_size": CODEBOOK_SIZE,
        "start_code": START_CODE,
        "max_text_tokens": MAX_TEXT_TOKENS,
        "max_frames": MAX_FRAMES,
        "train_last_n_layers": TRAIN_LAST_N_LAYERS,
    },
    final_dir / "qwen_mimi_frame_model.pt",
)
tokenizer.save_pretrained(final_dir)
print("saved:", final_dir)

## Generate Mimi Codes From Text

This cell generates code IDs only. To hear audio, pass the generated `[8, frames]` codes into the Mimi decoder in the later inference pipeline. Stage 1 does not yet learn an end-of-speech token, so generation uses a fixed `max_frames`; dialogue stages will add silence/speak control.

A good early model should generate diverse code IDs and avoid immediate collapse to one repeated token.

In [ ]:
@torch.no_grad()
def generate_mimi_codes(model, tokenizer, text: str, max_frames: int = 80, temperature: float = 0.8):
    model.eval()
    device = next(model.parameters()).device
    prompt = MimiFrameCollator._format_prompt(text)
    encoded = tokenizer(
        [prompt],
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_TOKENS,
        return_tensors="pt",
        add_special_tokens=True,
    )
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # prev_codes contains input frames. Position 0 starts with START_CODE.
    prev_codes = torch.full((1, K_CODEBOOKS, 1), START_CODE, dtype=torch.long, device=device)
    generated = []

    for _ in range(max_frames):
        code_mask = torch.ones((1, prev_codes.shape[-1]), dtype=torch.long, device=device)
        logits_by_cb = model.forward_logits(input_ids, attention_mask, prev_codes, code_mask)
        next_codes = []
        for cb in range(K_CODEBOOKS):
            logits = logits_by_cb[cb][:, -1, :] / max(temperature, 1e-6)
            probs = torch.softmax(logits.float(), dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)
            next_codes.append(next_id)
        next_frame = torch.stack(next_codes, dim=1)  # [1, K]
        generated.append(next_frame.squeeze(0).detach().cpu())
        prev_codes = torch.cat([prev_codes, next_frame.unsqueeze(-1)], dim=-1)

    return torch.stack(generated, dim=1)  # [K, T]

sample_codes = generate_mimi_codes(
    model,
    tokenizer,
    "mister quilter is the apostle of the middle classes and we are glad to welcome his gospel",
    max_frames=40,
)
print(sample_codes.shape)
print(sample_codes[:, :5])
print("unique cb0 tokens:", len(set(sample_codes[0].tolist())))

## What Good Stage 1 Looks Like

Expected successful outcomes:

```text
training/eval loss goes down
cb0 predictions become stable and diverse
generated [8, T] codes can be decoded by Mimi
short text prompts produce recognizable speech after decoding
no immediate token collapse
```

Expected limitations after Stage 1:

```text
not full duplex yet
not conversation-tuned yet
voice may be unstable
long generations may drift
```

Next steps after this notebook works:

```text
1. Move this model/trainer into repo scripts.
2. Add Mimi decode smoke test.
3. Train Stage 1 on more LibriSpeech Mimi rows.
4. Train Stage 2 on clean ZipVoice dialogue frame streams.
5. Train Stage 3 on overlap/full-duplex frame streams.
```